# Foundry IQ knowledge base with Web IQ MCP server

In Part 1, the knowledge base only queried pre-built search indexes. Now you'll add a third source type: an **MCP Server Knowledge Source** that connects to the new Web IQ service.

- **MCP Server Knowledge Source**: connects to any remote MCP endpoint. The KB calls it at query time like a tool.
- **Web IQ**: Microsoft's web grounding service exposed as an MCP server. Returns ranked, cited web results.

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has *almost* everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

‼️ You also need a `WEB_IQ_KEY` variable. This is only available for allow-listed customers at this time.

```shell
WEB_IQ_KEY="some-value-here"
```

Then run the cell below to load in those variables:

In [1]:
import json
import os

from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY")  # optional; unset uses managed identity
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureDeveloperCliCredential(tenant_id=AZURE_TENANT_ID)
print("Environment variables loaded")


Environment variables loaded


## Step 2: Create three knowledge sources

For this knowledge base, you'll first create three knowledge sources - the same two index-based sources as you made in the first notebook, plus an additional knowledge source for the Web IQ MCP server:

* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `web-knowledge-source`: Points to the remote MCP server for the Web IQ service, using key-based authentication and `web` tool.

In [2]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    McpServerAutoOutputParsing,
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
    McpServerTool,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME, WEB_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

web_knowledge_source = McpServerKnowledgeSource(
    name=WEB_KNOWLEDGE_SOURCE_NAME,
    description="Web IQ (live web search)",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://api.microsoft.ai/v3/mcp",
        authentication=McpServerStoredHeadersAuthentication(
            stored_headers_parameters=McpServerStoredHeadersParameters(
                {"headers": {"x-apikey": WEB_IQ_KEY}}
            )
        ),
        tools=[McpServerTool(name="web", output_parsing=McpServerAutoOutputParsing())],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=web_knowledge_source)
print("Knowledge sources created")


Knowledge sources created


## Step 3: Create the multi-source + web knowledge base

A knowledge base is the orchestration layer that combines:

1. Your data sources (the knowledge sources from Step 2)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format response

For this notebook, we are using an `outputMode` of "answerSynthesis" so that the attached LLM can also answer the original user query. We are also using a `retrievalReasoningEffort` of "low", which means that the LLM will be used for query planning and knowledge source selection.

In [3]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-web-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Contoso knowledge base combining HR and health documents with access to web search.",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use the HR and health indexes for company policy questions. Use Web IQ web tool for context from public webpages.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


Knowledge base created


## Step 4: Query the knowledge base

Ask a question where one half needs internal data and the other needs current external benchmarks:

* _"What mental health coverage does Contoso offer?"_ → should come from `healthdocs`
* _"What are industry benchmarks for mental health benefits at tech companies?"_ → should come from the **web** via Web IQ

The knowledge base uses agentic retrieval to:

1. Analyze the query and recognize two distinct topic domains
2. Decompose it into focused sub-queries
3. Route the internal question to the search indexes and the benchmark question to the web
4. Run searches concurrently across all three sources
5. Merge results using semantic re-ranking

In [4]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
web_knowledge_source_params = KnowledgeSourceParams(
    knowledge_source_name=WEB_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    kind="mcpServer",
)

question = (
    "A new hire wants to know if Contoso's benefits are competitive. "
    "What mental health coverage does Contoso offer according to our internal health plan docs? "
    "And search the web for current industry benchmarks for mental health benefits at tech companies."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params, web_knowledge_source_params],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))


I found relevant internal plan content, but the internal documents appear inconsistent, so I’d present this to the new hire as a plan-dependent summary rather than a single universal answer.[ref_id:1][ref_id:6][ref_id:11]

**Contoso internal mental health coverage from the health plan docs**  
- One internal document for **Northwind Standard** says the plan **does not cover any mental health and substance abuse coverage**, including **counselling** and **psychotherapy**.[ref_id:1]  
- But other internal **Northwind Standard** plan content says mental health care **is covered**, including **individual, family, and group psychotherapy** plus **psychiatric and psychological evaluation services**, generally with **in-network providers**.[ref_id:6]  
- That same Standard-plan content says there are exceptions: it **does not cover educational testing**, **assessment services**, the **cost of medications**, and **substance abuse treatment**.[ref_id:6]  
- Another Standard-plan section says **virtual care** is covered the same way as in-person care, including **mental health and substance abuse services** delivered by phone or video, as long as the provider is **in-network**.[ref_id:9]  

- Internal documents for **Northwind Health Plus** describe more comprehensive mental health coverage.[ref_id:11][ref_id:7]  
- Health Plus covers **counseling**, **psychiatric visits**, **therapy**, and **group therapy**.[ref_id:11]  
- Health Plus provides coverage for both **in-network and out-of-network** mental health services.[ref_id:11]  
- Health Plus covers both **inpatient and outpatient** mental health visits.[ref_id:11]  
- A compliance section for Health Plus says the plan includes **mental health and substance abuse services** and, under parity requirements, covers medically necessary **inpatient and outpatient services**, **medication management**, and **psychological and psychosocial therapies**.[ref_id:7][ref_id:2]  
- Another Health Plus section says **outpatient mental health services** from an **out-of-network provider** do **not require prior authorization**, though the provider must submit a claim for review and processing.[ref_id:14]  

**Bottom line from the internal docs**  
- Based on the documents retrieved, **Health Plus looks clearly competitive and comprehensive** for mental health, because it includes **counseling, therapy, psychiatric visits, group therapy, inpatient/outpatient care, some out-of-network coverage, and parity-compliant behavioral health coverage**.[ref_id:11][ref_id:7][ref_id:2][ref_id:14]  
- The **Standard** plan is **unclear in the retrieved materials** because one document says mental health is excluded entirely, while others say it is covered with notable limits.[ref_id:1][ref_id:6][ref_id:9]  
- I did **not** find an internal document in the retrieved set describing a dedicated **EAP** or a separate company mental health program; I only found medical-plan and wellness-stipend content.[ref_id:0][ref_id:6][ref_id:11]  

**Current public benchmarks for mental health benefits at tech companies / employers**  
- A 2024 Milliman employer pulse survey says **EAPs were offered by 95% of respondents** in 2024, making EAPs the **most common** mental health resource among employers surveyed.[ref_id:13]  
- The same 2024 survey says **74%** offered a **medical carrier’s embedded behavioral health program**.[ref_id:13]  
- It also says **47%** offered **third-party counseling and therapy** solutions.[ref_id:13]  
- The survey reports **16%** offered **onsite or near-site clinics** in 2024.[ref_id:13]  
- Milliman says the survey covered a **broad range of industries** representing roughly **700,000 lives**, so it is a useful employer benchmark, though not tech-only.[ref_id:13]  

- A public EAP benchmark article says **Technology & IT** organizations typically see **EAP utilization rates of 5% to 14%**.[ref_id:3]  
- The same source says global EAP utilization is typically **3% to 10%** for basic telephonic or online programs, **10% to 25%** for more proactive multi-channel programs, and **30%+** for best-in-class, culture-embedded programs.[ref_id:3]  
- That source also says the **EAPA minimum recommended baseline** is **6%** utilization.[ref_id:3]  
- It notes that in tech, **digital-first access** and **mental health advocacy** tend to raise utilization, while “**hustle culture**” can suppress it.[ref_id:3]  

- A 2025 AllOne Health trends report says that in 2024, **79% of all EAP referrals were for mental health counseling**, up from **71% in 2023**.[ref_id:8]  
- The same report says employees used multiple care channels, with **45%** preferring **in-person counseling**, **43%** using **video-based counseling**, and **12%** using **telephonic counseling**.[ref_id:8]  
- That suggests current market expectations include offering **both in-person and virtual counseling access**, not just a hotline.[ref_id:8]  

**How Contoso compares to those benchmarks**  
- If a Contoso employee is enrolled in **Northwind Health Plus**, Contoso appears aligned with common market features because the plan includes **counseling/therapy**, **psychiatric care**, **group therapy**, **inpatient/outpatient coverage**, and **out-of-network access**, which compares favorably with common employer offerings such as EAPs, embedded behavioral health programs, and third-party counseling solutions.[ref_id:11][ref_id:7][ref_id:2][ref_id:13]  
- The presence of **virtual care for mental health** in the Standard-plan materials also aligns with the broader benchmark trend toward **virtual and flexible access channels**.[ref_id:9][ref_id:8]  
- However, because the retrieved internal docs do not clearly show a dedicated **EAP** or unified enterprise mental health program, I cannot confirm that Contoso matches the **95% employer prevalence of EAP offerings** reported in the 2024 benchmark survey.[ref_id:13]  
- Also, because the Standard-plan documents conflict, I cannot confidently tell a new hire that **all Contoso medical options** are equally competitive for mental health coverage.[ref_id:1][ref_id:6][ref_id:9]  

**Practical summary for the new hire**  
- Contoso’s **Health Plus** option appears to offer **strong mental health coverage** with **counseling, therapy, psychiatry, group therapy, inpatient/outpatient treatment, and some out-of-network coverage**.[ref_id:11][ref_id:7][ref_id:2][ref_id:14]  
- Market benchmarks suggest competitive employers commonly offer **EAP access**, **embedded behavioral health through the medical carrier**, **third-party counseling/therapy**, and **virtual access options**.[ref_id:13][ref_id:8]  
- So the fairest answer is: **Contoso looks competitive if the employee enrolls in Health Plus**, but I would want the specific plan name confirmed before making a blanket statement, because the retrieved **Standard** plan documents conflict on whether mental health care is covered and on how broad that coverage is.[ref_id:1][ref_id:6][ref_id:9][ref_id:11]  

If you want, I can turn this into a short recruiter-friendly comparison blurb for candidates.[ref_id:11][ref_id:13]

### Review the activity log

For this activity log, you will see an "mcpServer" step for the call made to Web IQ, along with "searchIndex" steps for each query sent to a search index.

In [5]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1827,
    "outputTokens": 202,
    "modelName": "gpt-5.4",
    "elapsedMs": 3319
  },
  {
    "type": "searchIndex",
    "id": 1,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-07-24T00:24:34.1814381Z",
    "count": 4,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "mental health benefits",
      "filter": null,
      "semanticConfigurationName": "semantic-configuration",
      "sourceDataFields": [
        {
          "name": "snippet"
        },
        {
          "name": "uid"
        },
        {
          "name": "blob_path"
        }
      ],
      "searchFields": [
        {
          "name": "snippet"
        }
      ]
    }
  },
  {
    "type": "searchIndex",
    "id": 3,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-07-24T00:24:34.2423342Z",
    "count": 13,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "e

### Review the references

For the Web IQ knowledge source, the references have `type: "mcpServer"`. The article title, content, and URL are inside `sourceData.content`, which is a JSON string.

In [6]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

[
  {
    "type": "searchIndex",
    "id": "0",
    "activitySource": 3,
    "sourceData": {
      "uid": "810256edf976_aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL1BlcmtzUGx1cy5wZGY1_pages_0",
      "blob_path": "/hrdocs/PerksPlus.pdf",
      "snippet": "PerksPlus Health and Wellness \n\nReimbursement Program for \n\nContoso Employees \n\n  \n\n\n\nThis document contains information generated using a language model (Azure OpenAI). The \n\ninformation \n\ncontained in this document is only for demonstration purposes and does not reflect the \n\nopinions or \n\nbeliefs of Microsoft. Microsoft makes no representations or warranties of any kind, express or \n\nimplied, \n\nabout the completeness, accuracy, reliability, suitability or availability with respect to the \n\ninformation \n\ncontained in this document. \n\nAll rights reserved to Microsoft \n\n  \n\n\n\nOverview \n\nIntroducing PerksPlus - the ultimate benefits program designed to support the health 